# R(2+1)D-18 — Violence vs Non-Violence (MERGED_MINI)

Pipeline:
1. **Stage 1 – Feature extraction:** the Kinetics-400 pretrained R(2+1)D-18 backbone is frozen and used as a feature extractor; only a new binary head is trained.
2. **Stage 2 – Fine-tuning:** the last residual stage (`layer4`) is unfrozen and trained together with the head.

Each epoch draws a **new random clip** from every training video (random temporal start, random crop, random flip), so the model sees different clips each epoch.
For that reason features are not cached — the frozen backbone runs on the fly (no gradients, bf16), which is cheap.

Validation uses one fixed centre clip per video. The final evaluation averages `N_EVAL_CLIPS` evenly spaced clips per video.
Labels come from the folder names (`TRAIN/VAL` -> `VIOLENCE` / `NONVIOLENCE`). `TEST/` is treated as **unlabelled**: no metrics or confusion matrix are computed on it; the notebook only writes its predictions to `runs/test_predictions_*.csv`.

**Saved to `runs/`, and training is resumable.** After every epoch, `runs/{head,finetune}_last.pt` stores the model, optimizer, scheduler, history and RNG state. `runs/{head,finetune}_best.pt` stores the best weights by validation accuracy.
If training is interrupted, restart the kernel and **run all cells again**. Each stage continues from its last finished epoch. A stage that is already finished is not retrained; its best model is reloaded.
Set `RESUME = False`, or delete `runs/`, to start fresh.

## 1. Configuration — change paths here

In [ ]:
import os

# ---------------- PATHS (edit these if you move the notebook / dataset) ----------------
DATASET_ROOT  = "/home/haseebumer/content/data/MERGED_MINI"   # TRAIN/ and VAL/ have VIOLENCE/ + NONVIOLENCE/ folders; TEST/ is unlabelled
RUNS_DIR      = "./runs"                                      # weights, best models, resume checkpoints, plots, metrics
RESUME        = True                                          # True: continue from runs/*_last.pt if it exists

# ---------------- CLIP SAMPLING ----------------
CLIP_LEN     = 16          # frames per clip (what R(2+1)D-18 was pretrained with)
FRAME_STRIDE = 2           # take every 2nd frame -> a clip covers ~1 s of 30 fps video
RESIZE_HW    = (128, 171)  # resize (H, W) before cropping, as in torchvision's Kinetics recipe
CROP         = 112

# ---------------- TRAINING ----------------
# The GPU is shared (llama-server holds ~8.6 GB of 16 GB), so batches are sized for ~7 GB.
BATCH_SIZE_HEAD = 32       # stage 1: backbone frozen, no activations kept for backward
BATCH_SIZE_FT   = 16       # stage 2: layer4 activations kept for backward
NUM_WORKERS     = 8        # CPU decode processes, 1 FFmpeg thread each (12 was only ~10% faster and used more RAM)
PREFETCH        = 2        # batches queued per worker -> caps the host RAM held by the loader

HEAD_EPOCHS = 15
FT_EPOCHS   = 10
LR_HEAD     = 1e-3         # stage 1 head LR
LR_FT_HEAD  = 3e-4         # stage 2 head LR
LR_FT_L4    = 1e-4         # stage 2 layer4 LR
WEIGHT_DECAY = 1e-4

N_EVAL_CLIPS = 3           # clips averaged per video for the final metrics
SEED = 42

os.makedirs(RUNS_DIR, exist_ok=True)
assert os.path.isdir(DATASET_ROOT), f"DATASET_ROOT not found: {DATASET_ROOT}"

## 2. Imports and GPU setup

In [ ]:
import random, glob, time, copy
import numpy as np
import pandas as pd
import cv2
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision.models.video import r2plus1d_18, R2Plus1D_18_Weights
import matplotlib.pyplot as plt
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report
from tqdm.auto import tqdm

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.backends.cudnn.benchmark = True                 # fixed input size -> let cuDNN pick fastest conv3d kernels
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True
AMP_DTYPE = torch.bfloat16                            # RTX 50-series supports bf16 natively (no GradScaler needed)

print("torch", torch.__version__, "| device:", device)
if device.type == "cuda":
    free, total = torch.cuda.mem_get_info()
    print(torch.cuda.get_device_name(0), f"| free {free/2**30:.1f} GB / {total/2**30:.1f} GB")


def mem_status():
    """GPU memory held by this process (shows up in nvidia-smi) and host RAM still available."""
    ram = int(open("/proc/meminfo").read().split("MemAvailable:")[1].split()[0]) / 2**20
    return f"VRAM peak {torch.cuda.max_memory_reserved() / 2**30:.1f} GB | RAM free {ram:.1f} GB"

## 3. Build the file lists (labels from folder names: 0 = NONVIOLENCE, 1 = VIOLENCE; TEST has no labels)

In [ ]:
CLASSES = ["NONVIOLENCE", "VIOLENCE"]

def list_split(split):
    items = []
    for label, cls in enumerate(CLASSES):
        items += [(p, label) for p in sorted(glob.glob(os.path.join(DATASET_ROOT, split, cls, "*.mp4")))]
    return items

def list_test():
    """TEST/ is unlabelled (like unseen data in the real world): label is -1 and is never used."""
    return [(p, -1) for p in sorted(glob.glob(os.path.join(DATASET_ROOT, "TEST", "*.mp4")))]


train_items, val_items, test_items = list_split("TRAIN"), list_split("VAL"), list_test()
for name, it in [("TRAIN", train_items), ("VAL", val_items)]:
    labs = [l for _, l in it]
    print(f"{name:5s}: {len(it):5d} videos | violence {sum(labs):4d} | nonviolence {len(labs)-sum(labs):4d}")
print(f"TEST : {len(test_items):5d} videos | unlabelled (predictions only)")

## 4. Video clip dataset

- Decoding uses OpenCV with seeking, so long UCF videos are not decoded from the start.
- Video decoding has to run on the CPU (NVDEC was benchmarked: ~180 ms/clip vs ~40-70 ms on CPU, because a new hardware decoder is created for every short clip). Everything after that (normalisation, the model, the loss, the optimizer) runs on the GPU.
- Each video is decoded with **1 FFmpeg thread**. OpenCV's default of 16 threads per video, multiplied by all the workers, used about 5x more RAM and overloaded the CPU until the PC froze.
- Only one set of workers is alive at a time: train workers are reused across epochs, val/predict workers exist only while those loops run, and all are shut down if a cell is interrupted.
- **Train:** random temporal start, random crop and horizontal flip, so each epoch gets a different clip.
- **Eval:** evenly spaced clip(s) with a centre crop (deterministic).
- Workers return **uint8** tensors (4x less host-to-GPU traffic). Normalisation happens on the GPU.

In [ ]:
def open_video(path):
    # 1 decode thread per video: the default (16) costs ~5x more RAM per open video and,
    # summed over all workers, oversubscribes the 24 cores.
    return cv2.VideoCapture(path, cv2.CAP_FFMPEG, [cv2.CAP_PROP_N_THREADS, 1])


def read_clip(path, mode, clip_idx=0, n_clips=1):
    """Returns uint8 array (T, H, W, 3), T = CLIP_LEN, resized to RESIZE_HW."""
    cap = open_video(path)
    n_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    span = (CLIP_LEN - 1) * FRAME_STRIDE + 1
    max_start = max(n_frames - span, 0)
    if mode == "random":
        start = random.randint(0, max_start)
    else:                                   # evenly spaced; n_clips=1 -> centre clip
        start = int(round(max_start * (clip_idx + 1) / (n_clips + 1)))

    def grab(start):
        if start > 0:
            cap.set(cv2.CAP_PROP_POS_FRAMES, start)
        frames, i = [], 0
        while len(frames) < CLIP_LEN:
            ok, fr = cap.read()
            if not ok:
                break
            if i % FRAME_STRIDE == 0:
                fr = cv2.cvtColor(fr, cv2.COLOR_BGR2RGB)
                frames.append(cv2.resize(fr, (RESIZE_HW[1], RESIZE_HW[0]), interpolation=cv2.INTER_AREA))
            i += 1
        return frames

    frames = grab(start)
    if not frames and start > 0:            # seek failed (bad frame count) -> read from the beginning
        cap.release(); cap = open_video(path)
        frames = grab(0)
    cap.release()
    if not frames:
        print(f"WARNING: could not decode {path}")
        frames = [np.zeros((*RESIZE_HW, 3), np.uint8)]
    while len(frames) < CLIP_LEN:           # very short videos: repeat the last frame
        frames.append(frames[-1])
    return np.stack(frames)


class VideoClipDataset(Dataset):
    def __init__(self, items, train, n_clips=1):
        self.items, self.train, self.n_clips = items, train, n_clips

    def __len__(self):
        return len(self.items) * self.n_clips

    def __getitem__(self, idx):
        vid, clip_idx = divmod(idx, self.n_clips)
        path, label = self.items[vid]
        clip = read_clip(path, "random" if self.train else "uniform", clip_idx, self.n_clips)
        H, W = clip.shape[1:3]
        if self.train:
            y, x = random.randint(0, H - CROP), random.randint(0, W - CROP)
        else:
            y, x = (H - CROP) // 2, (W - CROP) // 2
        clip = clip[:, y:y + CROP, x:x + CROP]
        if self.train and random.random() < 0.5:
            clip = clip[:, :, ::-1]
        clip = torch.from_numpy(np.ascontiguousarray(clip)).permute(3, 0, 1, 2)   # (C, T, H, W) uint8
        return clip, label, vid


def worker_init(_):
    cv2.setNumThreads(1)                     # one decode thread per worker; parallelism comes from workers


def make_loader(ds, batch_size, shuffle, persistent=False):
    """persistent=True keeps workers alive between epochs (train only); otherwise they exit when the loop ends."""
    return DataLoader(ds, batch_size=batch_size, shuffle=shuffle, drop_last=shuffle,
                      num_workers=NUM_WORKERS, pin_memory=True, worker_init_fn=worker_init,
                      persistent_workers=persistent and NUM_WORKERS > 0,
                      prefetch_factor=PREFETCH if NUM_WORKERS > 0 else None)


def close_loader(loader):
    """Stops persistent workers now. Without this, an interrupted cell keeps them alive (Jupyter holds the
       traceback), and every re-run starts another set on top until RAM runs out."""
    it = getattr(loader, "_iterator", None)
    if it is not None and hasattr(it, "_shutdown_workers"):
        it._shutdown_workers()
    loader._iterator = None


MEAN = torch.tensor([0.43216, 0.394666, 0.37645], device=device).view(1, 3, 1, 1, 1)   # Kinetics stats
STD  = torch.tensor([0.22803, 0.22145, 0.216989], device=device).view(1, 3, 1, 1, 1)

def to_input(x):
    x = x.to(device, non_blocking=True).float().div_(255)
    return ((x - MEAN) / STD).contiguous(memory_format=torch.channels_last_3d)

## 5. Model: pretrained R(2+1)D-18 with a new binary head

In [ ]:
def build_model():
    model = r2plus1d_18(weights=R2Plus1D_18_Weights.KINETICS400_V1)
    model.fc = nn.Sequential(nn.Dropout(0.5), nn.Linear(model.fc.in_features, 1))   # 512 -> 1 logit
    return model.to(device).to(memory_format=torch.channels_last_3d)

n_pos = sum(l for _, l in train_items)
pos_weight = torch.tensor([(len(train_items) - n_pos) / max(n_pos, 1)], device=device)  # ~1.0 (balanced data)
criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
print("pos_weight:", round(pos_weight.item(), 3))


def set_mode(model, phase):
    """Frozen parts stay in eval mode (BatchNorm uses pretrained statistics)."""
    model.eval()
    model.fc.train()
    if phase == "finetune":
        model.layer4.train()
        for m in model.layer4.modules():     # small batches -> keep BN running stats frozen
            if isinstance(m, nn.BatchNorm3d):
                m.eval()


def run_epoch(model, loader, optimizer=None, phase=None):
    train = optimizer is not None
    if train:
        set_mode(model, phase)
    else:
        model.eval()
    tot_loss, correct, n = 0.0, 0, 0
    for x, y, _ in tqdm(loader, leave=False, desc="train" if train else "val"):
        x, y = to_input(x), y.to(device, non_blocking=True).float()
        with torch.set_grad_enabled(train), torch.autocast("cuda", dtype=AMP_DTYPE):
            logits = model(x).squeeze(1)
        loss = criterion(logits.float(), y)
        if train:
            optimizer.zero_grad(set_to_none=True)
            loss.backward()
            optimizer.step()
        tot_loss += loss.item() * y.size(0)
        correct += ((logits > 0).float() == y).sum().item()
        n += y.size(0)
    return tot_loss / n, correct / n


def atomic_save(obj, path):
    """Write to a temp file then rename, so an interrupted save never corrupts the checkpoint."""
    torch.save(obj, path + ".tmp")
    os.replace(path + ".tmp", path)


def get_rng_state():
    return {"python": random.getstate(), "numpy": np.random.get_state(), "torch": torch.get_rng_state(),
            "cuda": torch.cuda.get_rng_state_all() if torch.cuda.is_available() else None}


def set_rng_state(s):
    # RNG states must be CPU ByteTensors; torch.load(map_location=device) moved them to the GPU
    random.setstate(s["python"]); np.random.set_state(s["numpy"]); torch.set_rng_state(s["torch"].cpu())
    if s["cuda"] is not None and torch.cuda.is_available():
        torch.cuda.set_rng_state_all([t.cpu() for t in s["cuda"]])


def fit(model, optimizer, scheduler, epochs, batch_size, phase):
    """Trains `phase` for `epochs`, saving after every epoch to runs/:
         {phase}_last.pt -> model + optimizer + scheduler + history + RNG (used to resume)
         {phase}_best.pt -> best weights by val accuracy
       Re-running the cell resumes from the last finished epoch; a finished phase just reloads its best model."""
    last_path = os.path.join(RUNS_DIR, f"{phase}_last.pt")
    best_path = os.path.join(RUNS_DIR, f"{phase}_best.pt")
    hist = {k: [] for k in ["train_loss", "train_acc", "val_loss", "val_acc"]}
    best_acc, start_ep = -1.0, 1
    if RESUME and os.path.exists(last_path):
        ck = torch.load(last_path, map_location=device, weights_only=False)
        model.load_state_dict(ck["model"])
        optimizer.load_state_dict(ck["optimizer"])
        scheduler.load_state_dict(ck["scheduler"])
        set_rng_state(ck["rng"])
        hist, best_acc, start_ep = ck["history"], ck["best_acc"], ck["epoch"] + 1
        print(f"[{phase}] resumed from {last_path}: {ck['epoch']} epoch(s) done, best val acc {best_acc:.4f}")

    if start_ep <= epochs:
        train_loader = make_loader(VideoClipDataset(train_items, train=True), batch_size, shuffle=True, persistent=True)
        val_loader = make_loader(VideoClipDataset(val_items, train=False), batch_size * 2, shuffle=False)
    try:
        for ep in range(start_ep, epochs + 1):
            t0 = time.time()
            tr_loss, tr_acc = run_epoch(model, train_loader, optimizer, phase)
            va_loss, va_acc = run_epoch(model, val_loader)
            scheduler.step()
            for k, v in zip(hist, [tr_loss, tr_acc, va_loss, va_acc]):
                hist[k].append(v)
            flag = ""
            if va_acc > best_acc:
                best_acc = va_acc
                atomic_save({"model": model.state_dict(), "epoch": ep, "val_acc": va_acc, "phase": phase}, best_path)
                flag = "  <- best, saved"
            atomic_save({"model": model.state_dict(), "optimizer": optimizer.state_dict(),
                         "scheduler": scheduler.state_dict(), "rng": get_rng_state(),
                         "history": hist, "best_acc": best_acc, "epoch": ep}, last_path)
            print(f"[{phase}] epoch {ep:2d}/{epochs} | train loss {tr_loss:.4f} acc {tr_acc:.4f} | "
                  f"val loss {va_loss:.4f} acc {va_acc:.4f} | {time.time()-t0:.0f}s | {mem_status()}{flag}")
    finally:
        if start_ep <= epochs:
            close_loader(train_loader)
    best = torch.load(best_path, map_location=device, weights_only=False)
    model.load_state_dict(best["model"])
    print(f"Loaded best {phase} model from {best_path} (epoch {best['epoch']}, val acc {best['val_acc']:.4f})")
    return hist

## 6. Plotting and metric helpers

In [ ]:
def plot_history(hist, title, ft_start=None, save_name=None):
    ep = np.arange(1, len(hist["train_loss"]) + 1)
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    for ax, key, name in [(axes[0], "loss", "Loss"), (axes[1], "acc", "Accuracy")]:
        ax.plot(ep, hist[f"train_{key}"], "o-", label=f"Train {name.lower()}")
        ax.plot(ep, hist[f"val_{key}"], "s-", label=f"Val {name.lower()}")
        if ft_start is not None:
            ax.axvline(ft_start + 0.5, color="red", ls="--", lw=2, label="Fine-tuning started (layer4 unfrozen)")
        ax.set_xlabel("Epoch"); ax.set_ylabel(name); ax.set_title(f"{title} — {name}")
        ax.grid(alpha=0.3); ax.legend()
    plt.tight_layout()
    if save_name:
        plt.savefig(os.path.join(RUNS_DIR, save_name), dpi=150)
    plt.show()


@torch.no_grad()
def predict(model, items, n_clips=N_EVAL_CLIPS):
    """Averages the violence probability over n_clips evenly spaced clips per video."""
    model.eval()
    loader = make_loader(VideoClipDataset(items, train=False, n_clips=n_clips), BATCH_SIZE_HEAD * 2, shuffle=False)
    prob_sum, cnt = np.zeros(len(items)), np.zeros(len(items))
    for x, _, vid in tqdm(loader, leave=False, desc="predict"):
        with torch.autocast("cuda", dtype=AMP_DTYPE):
            p = torch.sigmoid(model(to_input(x)).squeeze(1).float()).cpu().numpy()
        np.add.at(prob_sum, vid.numpy(), p)
        np.add.at(cnt, vid.numpy(), 1)
    return prob_sum / cnt, np.array([l for _, l in items])


def report(model, items, split_name, stage_name):
    if not items:
        print(f"{split_name}: no labelled videos, skipped"); return None
    probs, y = predict(model, items)
    pred = (probs >= 0.5).astype(int)
    m = {"accuracy": accuracy_score(y, pred),
         "precision": precision_score(y, pred, zero_division=0),
         "recall": recall_score(y, pred, zero_division=0),
         "f1": f1_score(y, pred, zero_division=0)}
    print(f"===== {stage_name} | {split_name} ({len(items)} videos, {N_EVAL_CLIPS} clips each) =====")
    print(pd.DataFrame([m], index=[split_name]).round(4).to_string())
    print(classification_report(y, pred, target_names=CLASSES, digits=4, zero_division=0))
    cm = confusion_matrix(y, pred)
    fig, ax = plt.subplots(figsize=(4.5, 4))
    ax.imshow(cm, cmap="Blues")
    for i in range(2):
        for j in range(2):
            ax.text(j, i, cm[i, j], ha="center", va="center", fontsize=13,
                    color="white" if cm[i, j] > cm.max() / 2 else "black")
    ax.set_xticks([0, 1], CLASSES); ax.set_yticks([0, 1], CLASSES)
    ax.set_xlabel("Predicted"); ax.set_ylabel("True"); ax.set_title(f"{stage_name} — {split_name}")
    plt.tight_layout(); plt.show()
    return m


def predict_unlabelled(model, items, stage_name):
    """No labels, no metrics: saves the violence probability and predicted class of every video to a CSV."""
    if not items:
        print("TEST: no videos, skipped"); return None
    probs, _ = predict(model, items)
    df = pd.DataFrame({"filename": [os.path.basename(p) for p, _ in items],
                       "violence_prob": probs.round(4),
                       "prediction": np.where(probs >= 0.5, CLASSES[1], CLASSES[0])})
    out = os.path.join(RUNS_DIR, f"test_predictions_{stage_name.split()[0].lower()}.csv")
    df.to_csv(out, index=False)
    print(f"===== {stage_name} | TEST ({len(df)} videos) -> {out} =====")
    print(df["prediction"].value_counts().to_string())
    return df

## 7. Stage 1 — Feature extraction (frozen backbone, train only the head)

In [ ]:
model = build_model()
for p in model.parameters():
    p.requires_grad = False
for p in model.fc.parameters():
    p.requires_grad = True
print("trainable params:", sum(p.numel() for p in model.parameters() if p.requires_grad))

opt = torch.optim.AdamW(model.fc.parameters(), lr=LR_HEAD, weight_decay=WEIGHT_DECAY)
sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=HEAD_EPOCHS)
hist_head = fit(model, opt, sched, HEAD_EPOCHS, BATCH_SIZE_HEAD, "head")

### Stage 1 — accuracy & loss curves

In [ ]:
plot_history(hist_head, "Stage 1 (feature extraction)", save_name="curves_stage1.png")

### Stage 1 — accuracy, precision, recall, F1

In [ ]:
metrics = {}
metrics[("Feature extraction", "VAL")]  = report(model, val_items, "VAL", "Feature extraction")
test_pred_head = predict_unlabelled(model, test_items, "Feature extraction")

## 8. Stage 2 — Fine-tuning (unfreeze `layer4`, the last of the 4 residual stages)

Starts from the best Stage 1 checkpoint (already loaded in `model`).

In [ ]:
for p in model.layer4.parameters():
    p.requires_grad = True
print("trainable params:", sum(p.numel() for p in model.parameters() if p.requires_grad))

opt = torch.optim.AdamW([
    {"params": model.layer4.parameters(), "lr": LR_FT_L4},
    {"params": model.fc.parameters(),     "lr": LR_FT_HEAD},
], weight_decay=WEIGHT_DECAY)
sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=FT_EPOCHS)
torch.cuda.empty_cache()
hist_ft = fit(model, opt, sched, FT_EPOCHS, BATCH_SIZE_FT, "finetune")
torch.save(model.state_dict(), os.path.join(RUNS_DIR, "final_model_weights.pt"))   # plain state_dict of the best fine-tuned model

### Stage 1 + Stage 2 — accuracy & loss curves (red line = fine-tuning started)

In [ ]:
hist_all = {k: hist_head[k] + hist_ft[k] for k in hist_head}
plot_history(hist_all, "Feature extraction → Fine-tuning", ft_start=len(hist_head["train_loss"]),
             save_name="curves_stage1_stage2.png")

### Stage 2 — accuracy, precision, recall, F1

In [ ]:
metrics[("Fine-tuned (layer4)", "VAL")]  = report(model, val_items, "VAL", "Fine-tuned (layer4)")
test_pred_ft = predict_unlabelled(model, test_items, "Fine-tuned (layer4)")

## 9. Comparison: feature extraction vs fine-tuning

In [ ]:
summary = pd.DataFrame({k: v for k, v in metrics.items() if v is not None}).T
summary.index.names = ["stage", "split"]
summary.to_csv(os.path.join(RUNS_DIR, "metrics_summary.csv"))
summary.round(4)